# Dimension 2.4 - Cost-Effective Intervention Prioritization
ICER analysis, principal contradiction identification, sequential priority algorithm.

In [ ]:
import sys
sys.path.insert(0, "..")

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyArrowPatch

from hdi.config import PATHS, PARAMS
from hdi.data.loaders import load_paf_results, load_intervention_db

plt.rcParams["figure.dpi"] = 120
print("Imports OK")

## 1. Cost-Effectiveness Frontier

In [ ]:
# Build cost-effectiveness table from DCP3 reference data
# Data source: Disease Control Priorities, 3rd Edition (DCP3)
interventions = pd.DataFrame([
    {"intervention": "Childhood vaccination (EPI)", "cost_per_daly": 15, "dalys_averted_per_1M": 45_000, "category": "Infectious"},
    {"intervention": "Oral rehydration therapy", "cost_per_daly": 50, "dalys_averted_per_1M": 32_000, "category": "WaSH"},
    {"intervention": "Clean cookstoves", "cost_per_daly": 75, "dalys_averted_per_1M": 28_000, "category": "Air quality"},
    {"intervention": "Household water treatment", "cost_per_daly": 120, "dalys_averted_per_1M": 18_000, "category": "WaSH"},
    {"intervention": "Industrial emission standards", "cost_per_daly": 250, "dalys_averted_per_1M": 35_000, "category": "Air quality"},
    {"intervention": "Tobacco taxation (+50%)", "cost_per_daly": 30, "dalys_averted_per_1M": 22_000, "category": "NCD prevention"},
    {"intervention": "Vehicle emission standards", "cost_per_daly": 350, "dalys_averted_per_1M": 15_000, "category": "Air quality"},
    {"intervention": "Piped water infrastructure", "cost_per_daly": 500, "dalys_averted_per_1M": 40_000, "category": "WaSH"},
    {"intervention": "Urban green spaces", "cost_per_daly": 800, "dalys_averted_per_1M": 8_000, "category": "Urban planning"},
    {"intervention": "Air quality monitoring network", "cost_per_daly": 180, "dalys_averted_per_1M": 5_000, "category": "Surveillance"},
    {"intervention": "Sewage treatment plants", "cost_per_daly": 600, "dalys_averted_per_1M": 25_000, "category": "WaSH"},
    {"intervention": "Alcohol excise tax (+25%)", "cost_per_daly": 45, "dalys_averted_per_1M": 12_000, "category": "NCD prevention"},
])

# Sort by cost-effectiveness (ICER)
interventions = interventions.sort_values("cost_per_daly")
interventions["cumulative_dalys"] = interventions["dalys_averted_per_1M"].cumsum()
interventions["cumulative_cost"] = (interventions["cost_per_daly"] * interventions["dalys_averted_per_1M"]).cumsum()

print("Cost-Effectiveness Table (sorted by ICER):")
print(interventions[["intervention", "cost_per_daly", "dalys_averted_per_1M", "category"]].to_string(index=False))

# Plot cost-effectiveness frontier
fig, ax = plt.subplots(figsize=(10, 6))
colors = {"Infectious": "C0", "WaSH": "C1", "Air quality": "C2",
          "NCD prevention": "C3", "Urban planning": "C4", "Surveillance": "C5"}

for _, row in interventions.iterrows():
    ax.scatter(row["cost_per_daly"], row["dalys_averted_per_1M"],
              s=100, c=colors[row["category"]], zorder=5, edgecolors="k", lw=0.5)
    ax.annotate(row["intervention"], (row["cost_per_daly"], row["dalys_averted_per_1M"]),
               fontsize=7, ha="left", va="bottom", xytext=(5, 5),
               textcoords="offset points")

# Draw frontier (Pareto efficient interventions)
frontier = interventions.sort_values("dalys_averted_per_1M", ascending=False)
pareto = [frontier.iloc[0]]
for _, row in frontier.iloc[1:].iterrows():
    if row["cost_per_daly"] < pareto[-1]["cost_per_daly"]:
        pareto.append(row)
pareto_df = pd.DataFrame(pareto).sort_values("cost_per_daly")
ax.plot(pareto_df["cost_per_daly"], pareto_df["dalys_averted_per_1M"],
        "k--", alpha=0.5, lw=1, label="Efficiency frontier")

# Legend for categories
from matplotlib.lines import Line2D
handles = [Line2D([0], [0], marker="o", color="w", markerfacecolor=c,
                   markersize=8, label=cat) for cat, c in colors.items()]
ax.legend(handles=handles, loc="upper right", fontsize=8)
ax.set_xlabel("Cost per DALY averted (USD)")
ax.set_ylabel("DALYs averted per 1M population")
ax.set_title("Cost-Effectiveness Frontier (DCP3 Reference)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

## 2. Principal Contradiction Analysis (主次矛盾)

In [ ]:
# Identify the dominant risk factor per region using PAF results
# PAF = Population Attributable Fraction: proportion of disease burden
# attributable to each risk factor

paf = load_paf_results()

# Pivot: rows = regions, columns = risk factors, values = PAF
paf_wide = paf.pivot_table(
    index="region", columns="risk_factor", values="paf", aggfunc="mean"
).fillna(0)

print("PAF by Region and Risk Factor:")
print(paf_wide.round(3).to_string())
print()

# Principal contradiction: highest PAF risk factor per region
principal = paf_wide.idxmax(axis=1).to_frame("principal_risk")
principal["paf_value"] = paf_wide.max(axis=1)

# Secondary contradiction
secondary_paf = paf_wide.apply(
    lambda row: row.nlargest(2).iloc[-1] if len(row) >= 2 else np.nan, axis=1
)
secondary_risk = paf_wide.apply(
    lambda row: row.nlargest(2).index[-1] if len(row) >= 2 else "N/A", axis=1
)
principal["secondary_risk"] = secondary_risk
principal["secondary_paf"] = secondary_paf
principal["concentration_ratio"] = principal["paf_value"] / principal["secondary_paf"]

print("\nPrincipal Contradiction by Region:")
print(principal.round(3).to_string())
print()
print("Interpretation: concentration_ratio > 2 indicates a single dominant risk factor.")
print("Regions with ratio > 2 should prioritize the principal risk factor heavily.")

# Heatmap
fig, ax = plt.subplots(figsize=(10, 6))
im = ax.imshow(paf_wide.values, cmap="YlOrRd", aspect="auto")
ax.set_xticks(range(len(paf_wide.columns)))
ax.set_xticklabels(paf_wide.columns, rotation=45, ha="right", fontsize=8)
ax.set_yticks(range(len(paf_wide.index)))
ax.set_yticklabels(paf_wide.index, fontsize=8)

# Annotate cells
for i in range(len(paf_wide.index)):
    for j in range(len(paf_wide.columns)):
        val = paf_wide.values[i, j]
        if val > 0.01:
            ax.text(j, i, f"{val:.2f}", ha="center", va="center", fontsize=7,
                    color="white" if val > 0.3 else "black")

plt.colorbar(im, ax=ax, label="PAF")
ax.set_title("Population Attributable Fraction by Region and Risk Factor")
plt.tight_layout()
plt.show()

## 3. Priority Ranking by Income Group

In [ ]:
# Rank interventions by cost per DALY averted within income groups
# Apply income-group-specific willingness-to-pay thresholds

wtp_thresholds = {
    "Low income": 500,       # ~0.5x GDP per capita
    "Lower-middle": 1500,    # ~0.5x GDP per capita
    "Upper-middle": 5000,    # ~0.5x GDP per capita
    "High income": 50_000,   # ~1x GDP per capita
}

# Adjust cost-effectiveness by income group (costs scale with local prices)
cost_multipliers = {
    "Low income": 0.3,
    "Lower-middle": 0.5,
    "Upper-middle": 0.8,
    "High income": 1.5,
}

priority_tables = {}
for income_group, wtp in wtp_thresholds.items():
    mult = cost_multipliers[income_group]
    group_df = interventions.copy()
    group_df["adjusted_cost"] = group_df["cost_per_daly"] * mult
    group_df["cost_effective"] = group_df["adjusted_cost"] <= wtp
    group_df["priority_rank"] = group_df["adjusted_cost"].rank(method="min").astype(int)
    group_df = group_df.sort_values("priority_rank")
    priority_tables[income_group] = group_df

# Display priority rankings
for income_group, table in priority_tables.items():
    n_ce = table["cost_effective"].sum()
    print(f"\n{'='*70}")
    print(f"{income_group} (WTP threshold: ${wtp_thresholds[income_group]:,}/DALY)")
    print(f"{'='*70}")
    print(f"Cost-effective interventions: {n_ce}/{len(table)}")
    cols = ["priority_rank", "intervention", "adjusted_cost", "dalys_averted_per_1M", "cost_effective"]
    print(table[cols].to_string(index=False))

# Comparative bar chart
fig, axes = plt.subplots(2, 2, figsize=(14, 10))
for ax, (income_group, table) in zip(axes.flat, priority_tables.items()):
    sorted_t = table.sort_values("adjusted_cost")
    bar_colors = ["green" if ce else "gray" for ce in sorted_t["cost_effective"]]
    ax.barh(range(len(sorted_t)), sorted_t["adjusted_cost"], color=bar_colors, edgecolor="k", lw=0.5)
    ax.set_yticks(range(len(sorted_t)))
    ax.set_yticklabels(sorted_t["intervention"], fontsize=7)
    ax.axvline(wtp_thresholds[income_group], color="r", ls="--", lw=1.5, label=f"WTP=${wtp_thresholds[income_group]:,}")
    ax.set_xlabel("Cost per DALY averted (USD)")
    ax.set_title(income_group, fontsize=10)
    ax.legend(fontsize=7)

fig.suptitle("Intervention Priority Ranking by Income Group", fontsize=13, y=1.01)
plt.tight_layout()
plt.show()

## 4. Budget Allocation Pareto Frontier

In [ ]:
from pymoo.core.problem import ElementwiseProblem
from pymoo.algorithms.moo.nsga2 import NSGA2
from pymoo.optimize import minimize
from pymoo.visualization.scatter import Scatter

# Multi-objective optimization: maximize DALYs averted, minimize total cost
# Decision variables: fraction of max investment in each intervention [0, 1]
n_interventions = len(interventions)
max_budget = 50_000_000  # $50M total budget

# Cost to fully implement each intervention (per 1M population)
full_cost = (interventions["cost_per_daly"] * interventions["dalys_averted_per_1M"]).values

class BudgetAllocation(ElementwiseProblem):
    def __init__(self):
        super().__init__(
            n_var=n_interventions,
            n_obj=2,
            n_ieq_constr=1,
            xl=np.zeros(n_interventions),
            xu=np.ones(n_interventions),
        )

    def _evaluate(self, x, out, *args, **kwargs):
        # Objective 1: minimize negative DALYs averted (maximize DALYs averted)
        dalys = np.sum(x * interventions["dalys_averted_per_1M"].values)
        # Objective 2: minimize total cost
        cost = np.sum(x * full_cost)

        out["F"] = [-dalys, cost]
        # Constraint: total cost <= budget
        out["G"] = [cost - max_budget]

problem = BudgetAllocation()

algorithm = NSGA2(
    pop_size=200,
)

result = minimize(
    problem,
    algorithm,
    termination=("n_gen", 300),
    seed=42,
    verbose=False,
)

# Extract Pareto front
pareto_F = result.F.copy()
pareto_F[:, 0] = -pareto_F[:, 0]  # Convert back to positive DALYs

fig, ax = plt.subplots(figsize=(10, 6))
ax.scatter(pareto_F[:, 1] / 1e6, pareto_F[:, 0] / 1e3, s=15, alpha=0.7, c="steelblue")
ax.set_xlabel("Total Cost ($M)")
ax.set_ylabel("DALYs Averted (thousands per 1M pop)")
ax.set_title(f"Pareto Frontier: Budget Allocation (max ${max_budget/1e6:.0f}M)")
ax.grid(True, alpha=0.3)
plt.tight_layout()
plt.show()

# Show a few representative solutions
# Pick solutions at 25%, 50%, 75% of cost range
sorted_idx = np.argsort(pareto_F[:, 1])
picks = [sorted_idx[len(sorted_idx) // 4], sorted_idx[len(sorted_idx) // 2], sorted_idx[3 * len(sorted_idx) // 4]]

print("\nRepresentative Pareto-optimal allocations:")
for i, idx in enumerate(picks):
    x = result.X[idx]
    cost = pareto_F[idx, 1]
    dalys = pareto_F[idx, 0]
    print(f"\n--- Solution {i+1}: Cost=${cost/1e6:.1f}M, DALYs averted={dalys/1e3:.1f}k ---")
    for j, intv in enumerate(interventions["intervention"].values):
        if x[j] > 0.05:
            print(f"  {intv:40s}: {x[j]*100:.0f}% allocation")